In [19]:
import numpy as np
from scipy import integrate
from itertools import product

interval = [(0.1, 0.2), (0.8, 0.9)]

def create_combinations(tuple_list):
    combinations = product([0, 1], repeat=len(tuple_list))
    
    bounds = []
    for combo in combinations:
        new_list = [tuple_list[i][idx] for i, idx in enumerate(combo)]
        bounds.append(new_list)
    
    return bounds

def n_dim_integral(integrand, intervals, *args):
    res = 0 
    bounds = create_combinations(intervals)
    for bound in bounds:
        res += integrate.nquad(integrand, bound, args=args)[0]
    
    return res

In [20]:
def s2_bias(y1):
    return abs(y1 * (1 - s11 / (y1 * s11 + (1 - y1) * s01))) 

def s2_bias_sq(y1):
    return abs(y1 * (1 - s11 / (y1 * s11 + (1 - y1) * s01))) ** 2

###########################

def s2_vary(y1):
    p = y1 * s11 / (y1 * s11 + (1 - y1) * s01)
    var = p * (1 - p)
    return var

def s2_vary_sq(y1):
    p = y1 * s11 / (y1 * s11 + (1 - y1) * s01)
    var = p * (1 - p)
    return var ** 2

def s2_bias_vary(y1):
    bias = abs(y1 * (1 - s11 / (y1 * s11 + (1 - y1) * s01))) 

    p = y1 * s11 / (y1 * s11 + (1 - y1) * s01)
    var = p * (1 - p)
    return bias * var

###########################

def s2_vara(a, y0, y1):
    num = s01 * (1 - y1) * a + s11 * y1 * a
    denum = num + s00 * (1 - y0) * (1 - a) + s10 * y0 * (1 - a)

    p = num / denum
    var = p * (1 - p)

    return var

def s2_vara_sq(a, y0, y1):
    num = s01 * (1 - y1) * a + s11 * y1 * a
    denum = num + s00 * (1 - y0) * (1 - a) + s10 * y0 * (1 - a)

    p = num / denum
    var = p * (1 - p)

    return var ** 2

def s2_bias_vara(a, y0, y1):
    num = s01 * (1 - y1) * a + s11 * y1 * a
    denum = num + s00 * (1 - y0) * (1 - a) + s10 * y0 * (1 - a)

    p = num / denum
    var = p * (1 - p)

    bias = abs(y1 * (1 - s11 / (y1 * s11 + (1 - y1) * s01)))

    return var * bias

########################### 

def s2_vars(a, y0, y1):
    p = s01 * (1 - y1) * a + s11 * y1 * a + s00 * (1 - y0) * (1 - a) + s10 * y0 * (1 - a)
    var = p * (1 - p)

    return var

def s2_vars_sq(a, y0, y1):
    p = s01 * (1 - y1) * a + s11 * y1 * a + s00 * (1 - y0) * (1 - a) + s10 * y0 * (1 - a)
    var = p * (1 - p)

    return var ** 2

def s2_bias_vars(a, y0, y1):
    bias = abs(y1 * (1 - s11 / (y1 * s11 + (1 - y1) * s01))) 

    p = s01 * (1 - y1) * a + s11 * y1 * a + s00 * (1 - y0) * (1 - a) + s10 * y0 * (1 - a)
    var = p * (1 - p)

    return bias * var

In [22]:
s_list = [(0.1, 0.1, 0.1, 0.9),
        (0.9, 0.9, 0.9, 0.01),
        (0.1, 0.5, 0.5, 0.9),
        (0.9, 0.5, 0.5, 0.1),
        (0.1, 0.5, 0.9, 0.9),
        ]

for s in s_list:
    s00, s01, s10, s11 = s[0], s[1], s[2], s[3]
    n_mul = 1
    n_b = 1
    n_v = 1

    E_mul = (5 ** n_mul) * n_dim_integral(s2_bias_vary, [interval for _ in range(n_mul)])
    E_b = (5 ** n_b) * n_dim_integral(s2_bias, [interval for _ in range(n_b)])
    E_v = (5 ** n_v) * n_dim_integral(s2_vary, [interval for _ in range(n_v)])

    E_b_sq = (5 ** n_b) * n_dim_integral(s2_bias_sq, [interval for _ in range(n_b)])
    E_v_sq = (5 ** n_v) * n_dim_integral(s2_vary_sq, [interval for _ in range(n_v)])

    b_sig = np.sqrt(E_b_sq - E_b ** 2)
    v_sig = np.sqrt(E_v_sq - E_v ** 2)

    covY = (E_mul - E_b * E_v)
    rhoY =  covY / (b_sig * v_sig)

    #####################################

    n_mul = 3
    n_b = 1
    n_v = 3

    E_mul = (5 ** n_mul) * n_dim_integral(s2_bias_vara, [interval for _ in range(n_mul)])
    E_b = (5 ** n_b) * n_dim_integral(s2_bias, [interval for _ in range(n_b)])
    E_v = (5 ** n_v) * n_dim_integral(s2_vara, [interval for _ in range(n_v)])

    E_b_sq = (5 ** n_b) * n_dim_integral(s2_bias_sq, [interval for _ in range(n_b)])
    E_v_sq = (5 ** n_v) * n_dim_integral(s2_vara_sq, [interval for _ in range(n_v)])

    b_sig = np.sqrt(E_b_sq - E_b ** 2)
    v_sig = np.sqrt(E_v_sq - E_v ** 2)

    covA = (E_mul - E_b * E_v)
    rhoA =  covA / (b_sig * v_sig)

    #####################################

    n_mul = 3
    n_b = 1
    n_v = 3

    E_mul = (5 ** n_mul) * n_dim_integral(s2_bias_vars, [interval for _ in range(n_mul)])
    E_b = (5 ** n_b) * n_dim_integral(s2_bias, [interval for _ in range(n_b)])
    E_v = (5 ** n_v) * n_dim_integral(s2_vars, [interval for _ in range(n_v)])

    E_b_sq = (5 ** n_b) * n_dim_integral(s2_bias_sq, [interval for _ in range(n_b)])
    E_v_sq = (5 ** n_v) * n_dim_integral(s2_vars_sq, [interval for _ in range(n_v)])

    b_sig = np.sqrt(E_b_sq - E_b ** 2)
    v_sig = np.sqrt(E_v_sq - E_v ** 2)

    covS = (E_mul - E_b * E_v)
    rhoS =  covS / (b_sig * v_sig)

    #####################################

    print(f"s00: {s00}, s01: {s01}, s10: {s10}, s11:{s11}")

    print(f"CovY: {covY:.5f}, RhoY: {rhoY:.5f}")
    print(f"CovA: {covA:.5f}, RhoA: {rhoA:.5f}")
    print(f"CovS: {covS:.5f}, RhoS: {rhoS:.5f}")

    print('-' * 20)

s00: 0.1, s01: 0.1, s10: 0.1, s11:0.9
CovY: 0.01761, RhoY: 0.97986
CovA: 0.00020, RhoA: 0.01339
CovS: -0.00452, RhoS: -0.66002
--------------------
s00: 0.9, s01: 0.9, s10: 0.9, s11:0.01
CovY: 0.00901, RhoY: 0.96537
CovA: 0.00132, RhoA: 0.05194
CovS: 0.00683, RhoS: 0.63351
--------------------
s00: 0.1, s01: 0.5, s10: 0.5, s11:0.9
CovY: 0.00096, RhoY: 0.95377
CovA: -0.00001, RhoA: -0.00951
CovS: 0.00022, RhoS: 0.33125
--------------------
s00: 0.9, s01: 0.5, s10: 0.5, s11:0.1
CovY: 0.01071, RhoY: 0.98098
CovA: 0.00047, RhoA: 0.05786
CovS: -0.00134, RhoS: -0.37394
--------------------
s00: 0.1, s01: 0.5, s10: 0.9, s11:0.9
CovY: 0.00096, RhoY: 0.95377
CovA: -0.00001, RhoA: -0.00671
CovS: 0.00032, RhoS: 0.48644
--------------------


In [17]:
s_list = [(0.1, 0.1, 0.1, 0.9),
        (0.9, 0.9, 0.9, 0.1),
        (0.1, 0.5, 0.5, 0.9),
        (0.1, 0.9, 0.9, 0.1),
        ]

s = s_list[0]
s00, s01, s10, s11 = s[0], s[1], s[2], s[3]

for a in [0.1, 0.9]:
    for y0 in [0.1, 0.9]:
        for y1 in [0.1, 0.9]:
            bias = abs(y1 * (1 - s11 / (y1 * s11 + (1 - y1) * s01))) 
            ps = s01 * (1 - y1) * a + s11 * y1 * a + s00 * (1 - y0) * (1 - a) + s10 * y0 * (1 - a)
            pa = s01 * (1 - y1) * a + s11 * y1 * a / ps
            print(f"A:{a}, Y0: {y0}, Y1:{y1}, Bias : {bias:.2f}, ps:{ps:.2f} varS:{ps*(1-ps):.2f}, pa:{pa:.2f}, varA:{pa*(1-pa):.2f}")

A:0.1, Y0: 0.1, Y1:0.1, Bias : 0.40, ps:0.11 varS:0.10, pa:0.09, varA:0.08
A:0.1, Y0: 0.1, Y1:0.9, Bias : 0.09, ps:0.17 varS:0.14, pa:0.47, varA:0.25
A:0.1, Y0: 0.9, Y1:0.1, Bias : 0.40, ps:0.11 varS:0.10, pa:0.09, varA:0.08
A:0.1, Y0: 0.9, Y1:0.9, Bias : 0.09, ps:0.17 varS:0.14, pa:0.47, varA:0.25
A:0.9, Y0: 0.1, Y1:0.1, Bias : 0.40, ps:0.17 varS:0.14, pa:0.55, varA:0.25
A:0.9, Y0: 0.1, Y1:0.9, Bias : 0.09, ps:0.75 varS:0.19, pa:0.98, varA:0.02
A:0.9, Y0: 0.9, Y1:0.1, Bias : 0.40, ps:0.17 varS:0.14, pa:0.55, varA:0.25
A:0.9, Y0: 0.9, Y1:0.9, Bias : 0.09, ps:0.75 varS:0.19, pa:0.98, varA:0.02


In [6]:
# def bias_v1(x, y, z, w):
#     bias = abs((x - y) * (z - w) / (2 * (z + w)))
#     return bias

# def bias_v1_sq(x, y, z, w):
#     bias = abs((x - y) * (z - w) / (2 * (z + w)))
#     return bias ** 2

# def var_v1(z, w):
#     p = (z + w) / 2
#     var = p * (1 - p)
#     return var

# def var_v1_sq(z, w):
#     p = (z + w) / 2
#     var = p * (1 - p)
#     return var ** 2

# def mul_v1(x, y, z, w):
#     bias = abs((x - y) * (z - w) / (2 * (z + w)))

#     p = (z + w) / 2
#     var = p * (1 - p)

#     mul = bias * var
#     return mul

# n_mul = 4
# n_b = 4
# n_v = 2

# E_mul = (5 ** n_mul) * n_dim_integral(mul_v1, [interval for _ in range(n_mul)])
# E_b = (5 ** n_b) * n_dim_integral(bias_v1, [interval for _ in range(n_b)])
# E_v = (5 ** n_v) * n_dim_integral(var_v1, [interval for _ in range(n_v)])

# E_b_sq = (5 ** n_b) * n_dim_integral(bias_v1_sq, [interval for _ in range(n_b)])
# E_v_sq = (5 ** n_v) * n_dim_integral(var_v1_sq, [interval for _ in range(n_v)])

# b_sig = np.sqrt(E_b_sq - E_b ** 2)
# v_sig = np.sqrt(E_v_sq - E_v ** 2)

# res = (E_mul - E_b * E_v) / (b_sig * v_sig)